In [ ]:
import json
import sys

from pathlib import Path
from datetime import datetime, timezone

import websocket

# ==============================================================================
# Import Protobuf Decoder
# ==============================================================================

PROTO_PATH = Path(
    r"D:\Data Projects\MEXC API Architecture\websocket-proto"
)

sys.path.append(str(PROTO_PATH))

from PushDataV3ApiWrapper_pb2 import PushDataV3ApiWrapper

# ==============================================================================
# WebSocket Configuration
# ==============================================================================

WS_URL = "wss://wbs-api.mexc.com/ws"

SYMBOL = "ETHUSDT"

ENDPOINT = f"spot@public.aggre.deals.v3.api.pb@10ms@{SYMBOL}"

subscription = {
    "method": "SUBSCRIPTION",
    "params": [
        ENDPOINT
    ],
    "id": 1
}

received_time = lambda: datetime.now(
    timezone.utc
).strftime("%Y-%m-%d %H:%M:%S.%f UTC")

# ==============================================================================
# Connect
# ==============================================================================

ws = websocket.create_connection(WS_URL)

ws.send(json.dumps(subscription))


# ==============================================================================
# Human-Readable Number Formatter
# ==============================================================================

def format_notional(value: float) -> str:
    """
    Convert a numeric value into a human-readable format.

    Examples
    --------
    532.45          -> 532.45
    12543.82        -> 12.54K
    3841256.71      -> 3.84M
    9823456789.12   -> 9.82B
    """

    if value >= 1_000_000_000:
        return f"{value / 1_000_000_000:.2f}B"

    elif value >= 1_000_000:
        return f"{value / 1_000_000:.2f}M"

    elif value >= 1_000:
        return f"{value / 1_000:.2f}K"

    else:
        return f"{value:.2f}"


# ==============================================================================
# Receive Loop
# ==============================================================================

while True:

    message = ws.recv()

    if isinstance(message, str):
        continue

    wrapper = PushDataV3ApiWrapper()

    wrapper.ParseFromString(message)

    trades = wrapper.publicAggreDeals

    exchange_timestamp = datetime.fromtimestamp(
        wrapper.sendTime / 1000,
        tz=timezone.utc
    )

    print("\n" + "=" * 100)
    print("TRADE EXECUTIONS")
    print("=" * 100)

    print(f"Exchange Time : {exchange_timestamp}")
    print(f"Receive Time  : {received_time()}")
    print()

    print(
        f"{'Price':>12}"
        f"{'Quantity':>18}"
        f"{'Value (USDT)':>18}"
        f"{'Side':>12}"
    )

    for trade in trades.deals:

        price = float(trade.price)

        quantity = float(trade.quantity)

        value = price * quantity

        side = "BUY" if trade.tradeType == 1 else "SELL"

        print(
            f"{price:>12.2f}"
            f"{quantity:>18.5f}"
            f"{format_notional(value):>18}"
            f"{side:>12}"
        )